# RiskBands API UX Demo

Este notebook mostra a camada mais amigável da API do RiskBands no estilo sklearn/pandas.

Objetivos da demo:

- começar com um fluxo curto e memorável
- usar `DataFrame` com `y="target"`
- comparar `legacy` e `generalization_v1`
- inspecionar `summary()`, `binning_table()`, `score_details()` e `diagnostics()`
- gerar visuais comparativos com Plotly a partir de dados sintéticos


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display
from riskbands import Binner


In [ ]:
rng = np.random.default_rng(42)
n = 1200

df = pd.DataFrame(
    {
        "score": rng.normal(loc=0.0, scale=1.0, size=n),
        "utilization": rng.beta(2.5, 4.0, size=n),
        "month": rng.choice([202301, 202302, 202303, 202304, 202305, 202306], size=n),
    }
)

month_drift = (df["month"] - df["month"].min()) / 100
logit = -1.2 + 1.1 * df["score"] + 1.8 * df["utilization"] + 0.35 * month_drift
prob = 1.0 / (1.0 + np.exp(-logit))
df["target"] = (rng.random(n) < prob).astype(int)

df.head()


## 1. Fit no estilo dataframe-first

O fluxo recomendado agora é curto:

1. instanciar o `Binner`
2. chamar `fit(df, y="target", column="score", time_col="month")`
3. inspecionar `summary()` e `score_details()`
4. aplicar `transform(...)` quando quiser anexar o resultado ao `DataFrame`


In [ ]:
legacy_binner = Binner(
    strategy="supervised",
    max_n_bins=5,
    check_stability=True,
    score_strategy="legacy",
)

generalization_binner = Binner(
    strategy="supervised",
    max_n_bins=5,
    check_stability=True,
    score_strategy="generalization_v1",
    normalization_strategy="absolute",
    woe_shrinkage_strength=35.0,
)

legacy_binner.fit(df, y="target", column="score", time_col="month")
generalization_binner.fit(df, y="target", column="score", time_col="month")

friendly_summary = pd.concat(
    [
        legacy_binner.summary().assign(run="legacy"),
        generalization_binner.summary().assign(run="generalization_v1"),
    ],
    ignore_index=True,
)
friendly_summary


In [ ]:
score_bins = generalization_binner.transform(df["score"])
df_preview = df.assign(score_bin=score_bins)

binning_table = generalization_binner.binning_table()
score_details = generalization_binner.score_details()
temporal_diagnostics = generalization_binner.diagnostics(kind="bin")

display(df_preview.head())
display(binning_table)
display(score_details)
display(temporal_diagnostics.head())


## 2. Comparação visual com Plotly

Aqui a ideia é mostrar que a API amigável não esconde a arquitetura nova de scoring:

- `legacy` continua disponível
- `generalization_v1` continua controlável
- ambos podem ser comparados no mesmo fluxo de notebook


In [ ]:
comparison = pd.concat(
    [
        legacy_binner.score_details().assign(run="legacy"),
        generalization_binner.score_details().assign(run="generalization_v1"),
    ],
    ignore_index=True,
)

fig_score = px.bar(
    comparison,
    x="run",
    y="objective_preference_score",
    color="run",
    text="objective_score",
    title="Comparison-ready score by strategy",
)
fig_score.update_traces(texttemplate="raw=%{text:.3f}", textposition="outside")
fig_score.update_layout(showlegend=False, yaxis_title="objective_preference_score")
fig_score


In [ ]:
heatmap_df = temporal_diagnostics.pivot_table(
    index="bin",
    columns="month",
    values="event_rate",
    aggfunc="mean",
).sort_index(axis=1)

fig_heatmap = px.imshow(
    heatmap_df,
    aspect="auto",
    color_continuous_scale="YlOrRd",
    title="Event rate by bin and month",
    labels={"x": "month", "y": "bin", "color": "event_rate"},
)
fig_heatmap


In [ ]:
component_columns = [
    column
    for column in score_details.columns
    if column.startswith("objective_norm_")
]
component_frame = score_details.melt(
    id_vars=["variable", "score_strategy"],
    value_vars=component_columns,
    var_name="component",
    value_name="value",
)

fig_components = px.bar(
    component_frame,
    x="component",
    y="value",
    color="score_strategy",
    barmode="group",
    title="Normalized generalization objective components",
)
fig_components.update_layout(xaxis_tickangle=-35)
fig_components


## 3. Leituras rápidas

- `summary()` é a melhor porta de entrada para entender o resultado logo após o `fit`
- `binning_table()` deixa os cortes fáceis de inspecionar no notebook
- `score_details()` expõe o score final, os componentes e os pesos do objective
- `diagnostics(kind="bin")` ajuda a abrir estabilidade temporal sem vasculhar estruturas internas
- a camada pública ficou mais familiar, mas o controle fino de `score_strategy`, `score_weights`, `normalization_strategy` e `woe_shrinkage_strength` continua disponível
